In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib
import datetime

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables'

In [ ]:
orig_df = pd.read_excel(os.path.join(DIR, 'pubmed_results_Jamalah_713.xlsx'))
orig_df.head()

In [ ]:
orig2_df = pd.read_excel(os.path.join(DIR, r'fromJamalah9_6\pubmed_results_Jamalah_1556 - gpt_title_abstract_new_creteria.xlsx'))
orig2_df.head()

In [ ]:
orig2_df.columns

In [ ]:
orig2_df.drop(columns=['PMID', 'PMID.1', 'reasoning_title_abstract',
       'decision_title_abstract', 'reasoning_title', 'decision_title'], inplace=True)
orig2_df.columns

In [ ]:
orig_df.columns

In [ ]:
dates = orig_df.loc[orig_df.publication_date.notna(), 'publication_date']
dates = pd.to_datetime(dates)
orig_df.loc[orig_df.publication_date.notna(), 'year'] = dates.dt.year

In [ ]:
orig_df[['title', 'authors', 'year', 'abstract']].isna().sum()

In [ ]:
orig2_df[['title', 'authors', 'year', 'abstract']].isna().sum()

In [ ]:
orig2_df[['Title', 'Authors', 'Create Date', 'publication_date']].isna().sum()

In [ ]:
orig2_df['title'] = orig2_df['title'].fillna(orig2_df['Title'])
orig2_df['authors'] = orig2_df['authors'].fillna(orig2_df['Authors'])
orig2_df.drop(columns=['Title', 'Authors'], inplace=True)

In [ ]:
year = orig2_df['Create Date'].dt.year
orig2_df['year'] = orig2_df['year'].fillna(year)

In [ ]:
len(orig_df), len(orig2_df)

In [ ]:
orig2_df[['title', 'authors', 'year', 'abstract']].isna().sum()

In [ ]:
df = pd.concat([orig_df, orig2_df], axis=0)
df = df[df.title.notna()]
df.reset_index(inplace=True,drop=True)

In [ ]:
df['title_extract'] = df.title.map(extract_letters)

In [ ]:
df.loc[df.abstract.str.len() < 50, 'abstract'] = np.nan

In [ ]:
len(df)

In [ ]:
df['no_abstract'] = is_missing(df.abstract).astype(int)
df.head()

In [ ]:
df.sort_values('no_abstract', inplace=True)

In [ ]:
sum(dedup.no_abstract)

In [ ]:
sum(1 - df.no_abstract)

In [ ]:
eligibility_df = gpt.check_eligibility(df, criteria=gpt.JUMALAH_CRITERIA)

In [ ]:
dedup = eligibility_df.drop_duplicates('title_extract', keep='first')
len(dedup)

In [ ]:
dedup.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6\pubmed_and_jumalah_2232.xlsx', index=False)

In [ ]:
len(orig_df), len(orig_df.drop_duplicates())

In [ ]:
dedup.title_extract.isna().sum()

In [ ]:
tables = {}
for col in dedup.columns:
    if col == 'title_extract':
        continue
    df = dedup[['title_extract', col]]
    df = df[~is_missing(df[col])]
    df.drop_duplicates('title_extract', inplace=True)
    assert df.title_extract.isna().sum() == 0
    tables[col] = df

In [ ]:
merged = None
for key, df in tables.items():
    print(key)
    if merged is None:
        merged = df
        continue
    merged = pd.merge(merged, df, on='title_extract', how='outer')
len(merged)

In [ ]:
merged.isna().sum()

In [ ]:
merged.to_excel(os.path.join(DIR, 'smart_deduplicate.xlsx'))

In [ ]:
missing = is_missing(merged.year) | is_missing(merged.abstract)
has_pmid = ~is_missing(merged.PMID)
(has_pmid & missing).sum()

In [ ]:
titles = list(merged.loc[is_missing(merged.PMID), 'title'])
len(titles)

In [ ]:
parts = []
for title in tqdm(titles):
    part = pubmed.query(f'"{title}"', max_results=2)
    if len(part) != 1:
        continue
    part['orig_title'] = title
    parts.append(part)

In [ ]:
found_by_title_in_pubmed = pd.concat(parts, axis=0)
len(found_by_title_in_pubmed)

In [ ]:
found_by_title_in_pubmed.to_excel(os.path.join(DIR, 'by_title.xlsx'), index=False)

In [ ]:
found_by_title_in_pubmed['match'] = (found_by_title_in_pubmed.title.apply(extract_letters)
                                     == found_by_title_in_pubmed.orig_title.apply(extract_letters))
found_by_title_in_pubmed['match'].sum()

In [ ]:
found_by_title_in_pubmed = found_by_title_in_pubmed[found_by_title_in_pubmed['match']]
len(found_by_title_in_pubmed)

In [ ]:
merged.columns

In [ ]:
found_by_title_in_pubmed.columns

In [ ]:
found_by_title_in_pubmed['title_extract'] = found_by_title_in_pubmed.title.apply(extract_letters)

In [ ]:
found_by_title_in_pubmed = found_by_title_in_pubmed.drop(columns=['title', 'orig_title', 'match'])

In [ ]:
merged.drop(columns=['no_abstract'], inplace=True)

In [ ]:
merged_1 = pd.merge(merged, found_by_title_in_pubmed, on='title_extract', how='left', suffixes=('', '_pubmed'))
merged_1.head()

In [ ]:
merged_1.columns

In [ ]:
def fill_col(df, col1, col2):
    df.loc[df[col1] == '', col1] = np.nan
    df[col1] = df[col1].fillna(df[col2])
    df.drop(columns=[col2], inplace=True)

In [ ]:
merged_1.isna().sum()

In [ ]:
fill_col(merged_1, 'PMID', 'pubmed_id')
fill_col(merged_1, 'abstract', 'abstract_pubmed')
fill_col(merged_1, 'authors', 'authors_pubmed')

In [ ]:
merged_1.isna().sum()

In [ ]:
missing = is_missing(merged_1.year) | is_missing(merged_1.abstract)
has_pmid = ~is_missing(merged_1.PMID)
(has_pmid & missing).sum()

In [ ]:
missing_pmids = merged_1.loc[has_pmid & missing, ['PMID']]
from_pubmed = pubmed.add_lines_by_pubmed_id(missing_pmids, 'PMID')
len(from_pubmed)

In [ ]:
len(from_pubmed), from_pubmed.pubmed_id.nunique(), from_pubmed.PMID.nunique()

In [ ]:
from_pubmed = from_pubmed[from_pubmed.PMID == from_pubmed.pubmed_id]

In [ ]:
merged_2 = pd.merge(merged_1, from_pubmed, left_on='PMID', right_on='pubmed_id', how='left', suffixes=('', '_pubmed'))
len(merged_2)

In [ ]:
merged_2.drop(columns=['PMID_pubmed', 'pubmed_id'], inplace=True)

In [ ]:
for col in ['publication_date', 'abstract', 'authors']:
    fill_col(merged_2, col, f'{col}_pubmed')


In [ ]:
for col in merged_1.columns:
    print(col, is_missing(merged_1[col]).sum(), is_missing(merged_2[col]).sum())

In [ ]:
paul_df = pd.read_excel(os.path.join(DIR, 'paul74.xlsx'))
paul_df.columns

In [ ]:
paul_df['title_extract'] = paul_df.Title.apply(extract_letters)

In [ ]:
merged_2['is_in_paul'].sum()

In [ ]:
merged_2['is_in_paul'] = merged_2.title_extract.isin(paul_df.title_extract)
merged_2['is_in_paul'].sum()

In [ ]:
is_missing(merged_2.loc[merged_2.is_in_paul, 'abstract']).sum()

In [ ]:
merged_2.to_excel(os.path.join(DIR, 'deduplicate_updated_from_pubmed.xlsx'), index=False)

In [ ]:
merged_2 = pd.read_excel(os.path.join(DIR, 'deduplicate_updated_from_pubmed.xlsx'))

In [ ]:
not_selected = pd.read_excel(os.path.join(DIR, 'not_selected_titles_abstracts.xlsx'))
not_selected.columns

In [ ]:
not_selected['title_extract'] = not_selected['Title'].apply(extract_letters)

In [ ]:
merged_2['is_in_not_selected'] = merged_2.title_extract.isin(not_selected.title_extract)
merged_2['is_in_not_selected'].sum()

In [ ]:
merged_2.loc[merged_2['is_in_not_selected'], 'abstract']

In [ ]:
merged_2.loc[merged_2['is_in_not_selected'], 'is_in_paul']

In [ ]:
eligibility_df = gpt.check_eligibility(
    merged_2,
    criteria = gpt.PAUL_CRITERIA,
)
eligibility_df.decision.sum()

In [ ]:
eligibility_df.decision.sum()

In [ ]:
eligibility_df.to_excel(os.path.join(DIR, 'deduplicate_with_eligibility.xlsx'), index=False)